<a href="https://colab.research.google.com/github/michaelsteven1299/proyecto_michael-/blob/main/src/00_descargas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universidad Libre - Seccional Cali<br>Facultad de Ingeniería - Diplomado en Ciencia de Datos<br>(ↄ) Diego Fernando Marin, 2024

# 00_descargas
Plantilla para el desarrollo del proyecto del diplomado de Ciencia de Datos, aplicando buenas prácticas.

---

Este cuaderno representa el primer paso crucial en nuestro proceso de ciencia de datos: la obtención y almacenamiento de datos crudos. El principio fundamental aquí es preservar los datos en su estado original, sin modificaciones, para garantizar la reproducibilidad del análisis y mantener una referencia histórica confiable.

**Propósito:** Establecer el punto de partida del análisis mediante la recopilación y almacenamiento de datos crudos de múltiples fuentes, manteniendo la integridad y trazabilidad de la información original.

**Tareas habituales:**
- Configuración de conexiones a bases de datos SQL y ejecución de consultas
- Implementación de web scraping para extracción de datos de páginas web
- Desarrollo de scripts RPA (Robotic Process Automation) para automatizar descargas
- Autenticación y descarga de datos desde APIs
- Verificación de integridad de archivos descargados
- Documentación de fuentes, timestamps y métodos de obtención
- Establecimiento de estructura de carpetas para datos crudos
- Implementación de control de versiones para datos cuando sea aplicable

# DESCARGA DE DATOS DE YAHOO FINANCE

In [5]:
from google.colab import drive
import os

drive.mount('/content/drive')

RAW_PATH = "/content/drive/MyDrive/proyecto_oro/data/raw/"
os.makedirs(RAW_PATH, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import pandas as pd
import yfinance as yf

# Variables de mercado relacionadas con oro_COP:
# el propio oro, el tipo de cambio de mercado, petróleo (driver del peso
# colombiano), fuerza del dólar (DXY), apetito por riesgo (VIX) y tasa
# de referencia en USD (bono 10y)
tickers = {
    "GLD":      "oro_xauusd",
    "COP=X":    "usd_cop",
    "CL=F":     "wti_crudo",
    "^VIX":     "vix",
    "DX-Y.NYB": "dxy",
    "^TNX":     "bono_10y",
}

for ticker, nombre in tickers.items():
    try:
        df = yf.download(ticker, start="2021-01-01", end="2026-07-01",
                          auto_adjust=True, progress=False)

        if df.empty:
            print(f"⚠ Sin datos: {ticker}")
            continue

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.to_csv(RAW_PATH + f"{nombre}.csv")
        print(f"✓ {nombre}.csv — {len(df)} filas")

    except Exception as e:
        print(f"X {ticker}: {e}")

✓ oro_xauusd.csv — 1378 filas
✓ usd_cop.csv — 1428 filas
✓ wti_crudo.csv — 1380 filas
✓ vix.csv — 1379 filas
✓ dxy.csv — 1380 filas
✓ bono_10y.csv — 1378 filas


In [7]:
import requests

# Tasa Representativa del Mercado (TRM): tipo de cambio oficial COP/USD
# certificado por la Superintendencia Financiera de Colombia.
# Fuente: Datos Abiertos Colombia (API Socrata), dataset 32sa-8pi3
TRM_URL = "https://www.datos.gov.co/resource/32sa-8pi3.json"

params = {
    "$where": "vigenciadesde >= '2021-01-01T00:00:00' AND vigenciadesde <= '2026-07-01T00:00:00'",
    "$order": "vigenciadesde ASC",
    "$limit": 5000,
}

try:
    response = requests.get(TRM_URL, params=params, timeout=30)
    response.raise_for_status()

    df_trm = pd.DataFrame(response.json())
    df_trm["vigenciadesde"] = pd.to_datetime(df_trm["vigenciadesde"])
    df_trm["valor"] = df_trm["valor"].astype(float)

    df_trm = df_trm[["vigenciadesde", "valor"]].rename(
        columns={"vigenciadesde": "Date", "valor": "trm"}
    )
    df_trm.set_index("Date", inplace=True)

    df_trm.to_csv(RAW_PATH + "trm.csv")
    print(f"✓ trm.csv — {len(df_trm)} filas")

except Exception as e:
    print(f"X TRM: {e}")

✓ trm.csv — 1299 filas


In [8]:
# Verificamos que las descargas clave tengan rangos de fechas coherentes
for nombre in ["oro_xauusd", "usd_cop", "trm"]:
    df_check = pd.read_csv(RAW_PATH + f"{nombre}.csv", index_col=0, parse_dates=True)
    print(f"{nombre}: {len(df_check)} filas | {df_check.index.min().date()} → {df_check.index.max().date()}")

oro_xauusd: 1378 filas | 2021-01-04 → 2026-06-30
usd_cop: 1428 filas | 2021-01-01 → 2026-06-30
trm: 1299 filas | 2021-01-05 → 2026-07-01
